# Ps 07
**Author:** Erik An
**Date:** 2026-01-24

## Problem 2
**Task**

Compute the Taylor polynomial of order $n\in\mathbb{N}$ at $x_{0}=0$ of
1. $f(x)=e^{x}$,
2. $f(x)=\sin(x)$,
3. $f(x)=\cos(x)$.
In addition, write Julia code that takes $f$ and $x_{0}$ as input and returns the Taylor polynomial of order 2.
Visualize that your above computations are correct when $n=2$.

In [ ]:
using ForwardDiff
using Polynomials
using Plots

function get_taylor_polynomial(f, x_0 = 0.0, order = 2)
    coeff = zeros(Float64, order + 1)
    coeff[1] = f(x_0)

    if (order >= 1)
        coeff[2] = ForwardDiff.derivative(f, x_0)
    end

    if (order >= 2)
        f_prime = x -> ForwardDiff.derivative(f, x)
        second_deriv = ForwardDiff.derivative(f_prime, x_0)
        coeff[3] = second_deriv / factorial(2)
    end

    return Polynomial(coeff)
end

data = [
    (f = exp, label="exp(x)"),
    (f = sin, label="sin(x)"),
    (f = cos, label="cos(x)")
]

x_range = range(-2, 2, length=100)
p = plot(layout=(3,1), size=(600, 800), legend=:topleft)

for (i, item) in enumerate(data)
    poly_data = get_taylor_polynomial(item.f)
    plot!(p[i], x_range, item.f.(x_range), label="True $(item.label)", lw=3)
    plot!(p[i], x_range, poly_data.(x_range), label="Taylor Approx (n=2)", ls=:dash, lw=3)
end

savefig("./figs/p_2.png")

## Problem 3
**Task**

Determine those $s\in\mathbb{R}$, for which the integral
$$\int_{0}^{1}x^{s} \, dx$$
is well-defined.
In addition, illustrate your theoretical results by approximating the integral via the mid-point rule for a few distinct $s$ in Julia.

In [ ]:
using Printf

function M(f, P)
    area = 0.0
    for i in 1:length(P)-1
        mid = (P[i+1] + P[i]) / 2
        h = f(mid)
        w = P[i+1] - P[i]
        area += h * w
    end
    return area
end

n_intervals = 10_000
s_values = [0.5, 0.0, -0.5, -0.9, -1.0, -1.5]

println("Approximating integral of x^s from 0 to 1 with $n_intervals intervals:\n")
@printf("%-10s | %-15s | %-15s | %-15s\n", "s", "Approx", "Exact", "Error")
println("-"^62)

for s in s_values

    f(x) = x^s

    P = range(0, 1, n_intervals)
    approx = M(f, P)

    if s > -1
        exact = 1 / (s + 1)
        error = abs(approx - exact)
        @printf("%-10.2f | %-15.5f | %-15.5f | %-15.5e\n", s, approx, exact, error)
    else
        exact_str = "Diverges"
        @printf("%-10.2f | %-15.5f | %-15s | %-15s\n", s, approx, exact_str, "N/A")
    end
end